In [3]:
import os

# 1. Defina as variáveis ANTES de importar a biblioteca do Kaggle
os.environ['KAGGLE_USERNAME'] = "julianopantani"
os.environ['KAGGLE_KEY'] = "KGAT_3998666419eba98058d5df53fae6059e"

# 2. Agora sim fazemos o import (ele vai autenticar automaticamente ao importar)
from kaggle.api.kaggle_api_extended import KaggleApi

# 3. Inicializa a API
api = KaggleApi()

# Opcional: dependendo da versão, o authenticate nem é mais necessário pois ele 
# faz isso no import, mas deixamos aqui por garantia.
api.authenticate() 

dataset_slug = 'hernan4444/anime-recommendation-database-2020'
print(f"Iniciando o download do dataset: {dataset_slug}...")

api.dataset_download_files(
    dataset_slug, 
    path='./meus_dados_anime',
    unzip=True
)

print("Download concluído com sucesso!")

Iniciando o download do dataset: hernan4444/anime-recommendation-database-2020...
Dataset URL: https://www.kaggle.com/datasets/hernan4444/anime-recommendation-database-2020
Download concluído com sucesso!


In [4]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo gigante (pode levar uns 10 a 20 segundos dependendo do PC)
caminho_original = './meus_dados_anime/rating_complete.csv'
print("Carregando o arquivo gigante... Segura as pontas!")
df_ratings = pd.read_csv(caminho_original)

# 2. Descobre quem são todos os usuários únicos na base
usuarios_unicos = df_ratings['user_id'].unique()
print(f"Encontramos {len(usuarios_unicos)} usuários únicos no total.")

# 3. Define o tamanho do seu sample
QUANTIDADE_USERS = 2000

# 4. Sorteia os 2000 usuários de forma aleatória
print(f"Sorteando {QUANTIDADE_USERS} usuários aleatoriamente...")
np.random.seed(42) # Garante que o sorteio seja o mesmo toda vez que rodar
usuarios_sorteados = np.random.choice(usuarios_unicos, size=QUANTIDADE_USERS, replace=False)

# 5. Filtra a tabela gigante para manter APENAS as linhas dos sorteados
print("Filtrando as notas dos usuários sorteados...")
df_sample = df_ratings[df_ratings['user_id'].isin(usuarios_sorteados)]

# 6. Salva o resultado em um arquivo novo e leve
caminho_salvamento = './meus_dados_anime/rating_sample_2000.csv'
df_sample.to_csv(caminho_salvamento, index=False)

print("\n--- TUDO PRONTO! ---")
print(f"O arquivo original tinha {len(df_ratings)} linhas.")
print(f"O seu novo arquivo reduzido tem apenas {len(df_sample)} linhas.")
print(f"Salvo com sucesso em: {caminho_salvamento}")

Carregando o arquivo gigante... Segura as pontas!
Encontramos 310059 usuários únicos no total.
Sorteando 2000 usuários aleatoriamente...
Filtrando as notas dos usuários sorteados...

--- TUDO PRONTO! ---
O arquivo original tinha 57633278 linhas.
O seu novo arquivo reduzido tem apenas 375454 linhas.
Salvo com sucesso em: ./meus_dados_anime/rating_sample_2000.csv


In [5]:
import pandas as pd

# 1. Carrega o seu sample de 2000 usuários que criamos no passo anterior
print("Carregando o sample de notas...")
df_sample = pd.read_csv('./meus_dados_anime/rating_sample_2000.csv')

# 2. Carrega a tabela que traduz os IDs em nomes de animes
print("Carregando a lista de títulos de animes...")
df_anime_titles = pd.read_csv('./meus_dados_anime/anime.csv')

# Para economizar memória e deixar limpo, vamos manter apenas as colunas de ID e Título da tabela de animes
df_anime_titles = df_anime_titles[['MAL_ID', 'Name']]
# Vamos renomear as colunas para facilitar o cruzamento de dados
df_anime_titles.columns = ['anime_id', 'title']

# 3. Junta as duas tabelas (cruza o ID do sample com o ID da lista de títulos)
print("Cruzando as notas com os títulos correspondentes...")
df_final = pd.merge(df_sample, df_anime_titles, on='anime_id', how='inner')

# 4. Reorganiza as colunas exatamente na ordem que você pediu
df_final = df_final[['user_id', 'anime_id', 'title', 'rating']]

# Opcional: Vamos ordenar por usuário e depois por nota (da maior para a menor) para ficar bonito de ler
df_final = df_final.sort_values(by=['user_id', 'rating'], ascending=[True, False])

# 5. Salva o resultado final em um novo arquivo CSV muito mais amigável
caminho_salvamento = './meus_dados_anime/sample_com_titulos.csv'
df_final.to_csv(caminho_salvamento, index=False)

print("\n--- SUCESSO! ---")
print(f"Arquivo final gerado com {len(df_final)} linhas.")
print(f"Salvo em: {caminho_salvamento}")

# Mostra as primeiras 5 linhas para você ver como ficou
print("\nVeja uma amostra do resultado:")
print(df_final.head())

Carregando o sample de notas...
Carregando a lista de títulos de animes...
Cruzando as notas com os títulos correspondentes...

--- SUCESSO! ---
Arquivo final gerado com 375454 linhas.
Salvo em: ./meus_dados_anime/sample_com_titulos.csv

Veja uma amostra do resultado:
    user_id  anime_id                                              title  \
1        17      6682                                             11eyes   
9        17     19221  Ore no Nounai Sentakushi ga, Gakuen Love Comed...   
11       17     20939  Ore no Nounai Sentakushi ga, Gakuen Love Comed...   
18       17      5530                                     Pandora Hearts   
41       17     10049                  Nurarihyon no Mago: Sennen Makyou   

    rating  
1       10  
9       10  
11      10  
18      10  
41      10  


In [5]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
import scipy.sparse as sp

# ========================= CONFIGURAÇÕES =========================
# Ajuste aqui conforme sua necessidade
MIN_RATINGS = 3          # Animes com menos de 100 avaliações serão removidos
THRESHOLD_NOTA_ALTA = 7    # Nota mínima considerada "alta"

print("🚀 Iniciando processamento da Matrizona com filtro de popularidade...\n")

# 1. Carregar o dataset
print("📂 Carregando o dataset...")
df = pd.read_csv('./meus_dados_anime/sample_com_titulos.csv')

# Garantir que a coluna rating seja numérica
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Remover linhas sem rating (opcional, mas recomendado)
df = df.dropna(subset=['rating'])

print(f"✅ Dataset carregado: {len(df):,} avaliações")

# ===================== FILTRAGEM DE ANIMES POPULARES =====================
print(f"🔍 Filtrando animes com pelo menos {MIN_RATINGS} avaliações...")

# Contar quantas avaliações cada anime tem
anime_ratings_count = df.groupby('title')['rating'].count()

# Selecionar apenas animes populares
animes_populares = anime_ratings_count[anime_ratings_count >= MIN_RATINGS].index

# Filtrar o DataFrame original
df_filtrado = df[df['title'].isin(animes_populares)].copy()

print(f"   → Animes antes: {len(anime_ratings_count)}")
print(f"   → Animes após filtro: {len(animes_populares)}")
print(f"   → Redução: {len(anime_ratings_count) - len(animes_populares)} animes removidos")

# ===================== CRIAÇÃO DE MAPEAMENTOS =====================
print("🗺️ Criando mapeamentos de usuários e animes...")

user_ids = df_filtrado['user_id'].unique()
anime_titles = df_filtrado['title'].unique()

# Dicionários para converter para índices numéricos (necessário para matriz esparsa)
user_to_idx = {user: i for i, user in enumerate(user_ids)}
anime_to_idx = {anime: i for i, anime in enumerate(anime_titles)}

n_users = len(user_ids)
n_animes = len(anime_titles)

print(f"📊 Dimensões finais: {n_users:,} usuários × {n_animes:,} animes")

# ===================== CRIAÇÃO DA MATRIZ ESPARSA =====================
print("⚡ Criando matriz esparsa User x Anime...")

# Pegar os índices numéricos
rows = df_filtrado['user_id'].map(user_to_idx).values
cols = df_filtrado['title'].map(anime_to_idx).values
data = df_filtrado['rating'].values

# Criar matriz esparsa no formato COO (eficiente para construção)
matriz_rating = coo_matrix((data, (rows, cols)), 
                          shape=(n_users, n_animes)).tocsr()

print("✅ Matriz esparsa criada com sucesso")

# ===================== MATRIZES BINÁRIAS =====================
print("🔢 Criando matrizes binárias...")

# Matriz A: Quem assistiu (1 = assistiu, 0 = não)
matriz_assistidos = (matriz_rating > 0).astype(np.int8)

# Matriz B: Quem deu nota alta (>= 7)
matriz_notas_altas = (matriz_rating >= THRESHOLD_NOTA_ALTA).astype(np.int8)

# ===================== CÁLCULO DAS CO-OCORRÊNCIAS =====================
print("🔄 Calculando co-ocorrências (multiplicação esparsa)...")

# Multiplicação de matrizes esparsas = muito mais rápido
co_assistidos = matriz_assistidos.T.dot(matriz_assistidos)
co_notas_altas = matriz_notas_altas.T.dot(matriz_notas_altas)

# Converter para dense apenas no final (float32 para economizar memória)
co_assistidos = co_assistidos.toarray().astype(np.float32)
co_notas_altas = co_notas_altas.toarray().astype(np.float32)

# ===================== CÁLCULO DA MATRIZ FINAL =====================
print("📈 Calculando porcentagens finais...")

# Divisão segura (evita divisão por zero)
matrizona_final = np.divide(
    co_notas_altas,
    co_assistidos,
    where=co_assistidos > 0,
    out=np.full_like(co_notas_altas, np.nan)
)

# Zerar diagonal principal (um anime não deve ser comparado consigo mesmo)
np.fill_diagonal(matrizona_final, np.nan)

# ===================== SALVAR RESULTADO =====================
print("💾 Salvando Matrizona...")

matrizona_df = pd.DataFrame(matrizona_final,
                           index=anime_titles,
                           columns=anime_titles)

caminho_saida = './meus_dados_anime/matrizona_porcentagens_filtrada.csv'
matrizona_df.to_csv(caminho_saida)

print("\n" + "="*60)
print("🎉 MALUQUICE CONCLUÍDA COM SUCESSO!")
print("="*60)
print(f"📁 Arquivo salvo em: {caminho_saida}")
print(f"📐 Tamanho final: {n_animes} x {n_animes} animes")
print(f"🔢 Mínimo de avaliações por anime: {MIN_RATINGS}")
print("="*60)

🚀 Iniciando processamento da Matrizona com filtro de popularidade...

📂 Carregando o dataset...
✅ Dataset carregado: 375,454 avaliações
🔍 Filtrando animes com pelo menos 3 avaliações...
   → Animes antes: 10339
   → Animes após filtro: 7258
   → Redução: 3081 animes removidos
🗺️ Criando mapeamentos de usuários e animes...
📊 Dimensões finais: 2,000 usuários × 7,258 animes
⚡ Criando matriz esparsa User x Anime...
✅ Matriz esparsa criada com sucesso
🔢 Criando matrizes binárias...
🔄 Calculando co-ocorrências (multiplicação esparsa)...
📈 Calculando porcentagens finais...
💾 Salvando Matrizona...

🎉 MALUQUICE CONCLUÍDA COM SUCESSO!
📁 Arquivo salvo em: ./meus_dados_anime/matrizona_porcentagens_filtrada.csv
📐 Tamanho final: 7258 x 7258 animes
🔢 Mínimo de avaliações por anime: 3
